# Moukthika — Hyperparameter Tuning

**Scope:** Randomized search over promising TF-IDF classifiers for Safe-Guard prompt-injection detection.

- Dataset: `xTRam1/safe-guard-prompt-injection` (pinned revision in `src/hyperparameter_tuning/config.py`)
- Official **test** split held out
- Validate from **train** only
- Primary metric: **recall on injection label 1**

Sibling streams own model-family selection and final training/optimization.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.hyperparameter_tuning.config import SEARCH_SPACES, DATASET_NAME, DATASET_REVISION, PRIMARY_SCORING
from src.hyperparameter_tuning.tune import run

print("Dataset:", DATASET_NAME, "@", DATASET_REVISION)
print("Primary scoring:", PRIMARY_SCORING)
print("Models:", list(SEARCH_SPACES))

## Search space

See `src/hyperparameter_tuning/README.md` and `config.py` for the full distributions.
Shared TF-IDF knobs: `ngram_range`, `min_df`, `max_df`, `sublinear_tf`.
Per-model: `C` / `alpha` / tree depth / `class_weight`, etc.

In [ ]:
# Full run (~minutes). For a smoke test use a subset + smaller n_iter:
# results_path = run(models=["logistic_regression", "multinomial_nb"], n_iter=5)

results_path = run()
results_path

In [ ]:
import json
from IPython.display import Markdown, display

artifacts = results_path.parent
summary = (artifacts / "tuning_summary.md").read_text(encoding="utf-8")
display(Markdown(summary))

payload = json.loads(results_path.read_text(encoding="utf-8"))
best = payload["best_overall"]
print("Best model:", best["model"])
print("Best params:", json.dumps(best["best_params"], indent=2))
print("Holdout metrics:", json.dumps(best["holdout_from_train"], indent=2))